# Customer Segmentation Using Clustering

This notebook reproduces the Lesson 6 wholesale customer segmentation pipeline using K-Means and adds Agglomerative Clustering as a second clustering method.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

RANDOM_STATE = 42

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Dataset

Load `raw_wholesale_customers.csv`, show the first rows, and inspect dataset information.

In [2]:
df = pd.read_csv("raw_wholesale_customers.csv")

print("=== FIRST 5 ROWS ===")
display(df.head())

print("\n=== DATASET INFO ===")
df.info()

print("\nDataset Shape:", df.shape)

=== FIRST 5 ROWS ===


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185



=== DATASET INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    int64
 7   Delicassen        440 non-null    int64
 8   Cluster           440 non-null    int32
 9   Agg_Cluster       440 non-null    int64
dtypes: int32(1), int64(9)
memory usage: 32.8 KB

Dataset Shape: (440, 8)


## 3. Select Features

Only the six spending columns are used for clustering. `Channel` and `Region` are not used as clustering features because they are categorical identifiers, not customer spending amounts.

In [3]:
features = [
    "Fresh",
    "Milk",
    "Grocery",
    "Frozen",
    "Detergents_Paper",
    "Delicassen"
]

X = df[features].copy()

print("Selected clustering features:")
print(features)

print("\nFeature Shape:", X.shape)
display(X.head())

Selected clustering features:
['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

Feature Shape: (440, 6)


,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,12669,9656,7561,214,2674,1338
1,7057,9810,9568,1762,3293,1776
2,6353,8808,7684,2405,3516,7844
3,13265,1196,4221,6404,507,1788
4,22615,5410,7198,3915,1777,5185


## 4. Apply IQR Capping

IQR capping clips extreme values using `k=1.5`. Rows are not deleted; values below the lower bound or above the upper bound are capped.

In [4]:
def iqr_cap(data, columns, k=1.5):
    capped_data = data.copy()
    cap_summary = []

    for column in columns:
        Q1 = capped_data[column].quantile(0.25)
        Q3 = capped_data[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - k * IQR
        upper_bound = Q3 + k * IQR

        values_capped = (
            (capped_data[column] < lower_bound) |
            (capped_data[column] > upper_bound)
        ).sum()

        capped_data[column] = capped_data[column].clip(
            lower=lower_bound,
            upper=upper_bound
        )

        cap_summary.append([
            column,
            Q1,
            Q3,
            lower_bound,
            upper_bound,
            values_capped
        ])

    cap_summary_df = pd.DataFrame(
        cap_summary,
        columns=[
            "Feature",
            "Q1",
            "Q3",
            "Lower Bound",
            "Upper Bound",
            "Values Capped"
        ]
    )

    return capped_data, cap_summary_df


X_capped, cap_summary_df = iqr_cap(X, features, k=1.5)

print("IQR capping completed successfully.")
print("Rows before capping:", X.shape[0])
print("Rows after capping :", X_capped.shape[0])

display(cap_summary_df.round(2))
display(X_capped.head())

IQR capping completed successfully.
Rows before capping: 440
Rows after capping : 440


,Feature,Q1,Q3,Lower Bound,Upper Bound,Values Capped
0,Fresh,3127.75,16933.75,-17581.25,37642.75,20
1,Milk,1533.00,7190.25,-6952.88,15676.12,28
2,Grocery,2153.00,10655.75,-10601.12,23409.88,24
3,Frozen,742.25,3554.25,-3475.75,7772.25,43
4,Detergents_Paper,256.75,3922.00,-5241.12,9419.88,30
5,Delicassen,408.25,1820.25,-1709.75,3938.25,27


,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,12669.0,9656.0,7561.0,214.0,2674.0,1338.00
1,7057.0,9810.0,9568.0,1762.0,3293.0,1776.00
2,6353.0,8808.0,7684.0,2405.0,3516.0,3938.25
3,13265.0,1196.0,4221.0,6404.0,507.0,1788.00
4,22615.0,5410.0,7198.0,3915.0,1777.0,3938.25


## 5. Scale Features

StandardScaler is applied before clustering so that large spending columns do not dominate the distance calculations.

In [5]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_capped)

print("Feature scaling completed successfully.")
print("Scaled Data Shape:", X_scaled.shape)

print("\nMean after scaling:")
print(X_scaled.mean(axis=0).round(4))

print("\nStandard deviation after scaling:")
print(X_scaled.std(axis=0).round(4))

Feature scaling completed successfully.
Scaled Data Shape: (440, 6)

Mean after scaling:
[-0. -0.  0.  0. -0.  0.]

Standard deviation after scaling:
[1. 1. 1. 1. 1. 1.]


## 6. Elbow Method

K-Means is trained for `k=1` to `k=10`, and SSE/inertia is printed for each value.

In [6]:
print("\n=== ELBOW METHOD (SSE per k) ===")

sse = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        n_init="auto",
        random_state=RANDOM_STATE
    )

    kmeans.fit(X_scaled)

    sse.append(kmeans.inertia_)

    print(f"k={k} -> SSE={kmeans.inertia_:.2f}")


=== ELBOW METHOD (SSE per k) ===
k=1 -> SSE=2640.00
k=2 -> SSE=1728.19
k=3 -> SSE=1363.45
k=4 -> SSE=1202.41
k=5 -> SSE=1070.15
k=6 -> SSE=964.76
k=7 -> SSE=921.14
k=8 -> SSE=776.63
k=9 -> SSE=726.88
k=10 -> SSE=707.41


## 7. Plot Elbow Curve

In [7]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, 11),
    sse,
    marker="o"
)

plt.xlabel("Number of Clusters (k)")
plt.ylabel("SSE (Inertia)")
plt.title("Elbow Method for K-Means")

plt.xticks(range(1, 11))
plt.grid()

plt.show()

## 8. Train K-Means

The lesson requires `KMeans(n_clusters=5, n_init="auto", random_state=42)`.

In [8]:
kmeans = KMeans(
    n_clusters=5,
    n_init="auto",
    random_state=RANDOM_STATE
)

kmeans_labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = kmeans_labels

print("K-Means training completed successfully.")

print("\nCluster Counts:")
print(df["Cluster"].value_counts().sort_index())

K-Means training completed successfully.

Cluster Counts:
Cluster
0     76
1    191
2     25
3     88
4     60


## 9. Evaluate K-Means

In [9]:
kmeans_silhouette = silhouette_score(
    X_scaled,
    kmeans_labels
)

kmeans_db = davies_bouldin_score(
    X_scaled,
    kmeans_labels
)

print("=== K-MEANS PERFORMANCE ===")

print(f"Silhouette Score: {kmeans_silhouette:.4f}")
print(f"Davies-Bouldin Index: {kmeans_db:.4f}")

=== K-MEANS PERFORMANCE ===
Silhouette Score: 0.2831
Davies-Bouldin Index: 1.2701


## 10. Cluster Centers in Original Spend Units

In [10]:
centers_original = scaler.inverse_transform(
    kmeans.cluster_centers_
)

centers_df = pd.DataFrame(
    centers_original,
    columns=features
)

centers_df.index.name = "Cluster"

print("=== CLUSTER CENTERS IN ORIGINAL SPEND UNITS ===")

display(centers_df.round(2))

=== CLUSTER CENTERS IN ORIGINAL SPEND UNITS ===


,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
Cluster,,,,,,
0,9202.67,6833.30,9104.12,1326.16,3280.12,1871.76
1,8376.23,2150.65,3160.63,1646.33,779.25,674.02
2,17461.54,13805.60,17524.12,4120.57,5460.56,3583.64
3,22346.70,3409.14,3969.33,5819.60,583.07,1566.95
4,4916.98,10768.85,18350.13,1212.37,7780.02,981.37


## 11. Research and Train a Second Clustering Algorithm

### Second Clustering Algorithm: Agglomerative Clustering

I selected **Agglomerative Clustering** as the second clustering algorithm. Agglomerative Clustering is a hierarchical clustering method that starts by treating every customer as an individual cluster. It repeatedly merges the most similar clusters until the required number of clusters is reached.

This algorithm fits wholesale customer segmentation because customers with similar purchasing patterns can be grouped based on their distances in the scaled feature space. Unlike K-Means, Agglomerative Clustering does not depend on randomly initialized centroids.

Source used for research: Scikit-learn documentation for Agglomerative Clustering: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html


In [11]:
agg_model = AgglomerativeClustering(
    n_clusters=5
)

agg_labels = agg_model.fit_predict(X_scaled)

df["Agg_Cluster"] = agg_labels

print("Agglomerative Clustering completed successfully.")

print("\nCluster Counts:")
print(df["Agg_Cluster"].value_counts().sort_index())

Agglomerative Clustering completed successfully.

Cluster Counts:
Agg_Cluster
0     70
1     72
2    164
3     55
4     79


## 12. Evaluate Agglomerative Clustering

In [12]:
agg_silhouette = silhouette_score(
    X_scaled,
    agg_labels
)

agg_db = davies_bouldin_score(
    X_scaled,
    agg_labels
)

print("=== AGGLOMERATIVE CLUSTERING PERFORMANCE ===")

print(f"Silhouette Score: {agg_silhouette:.4f}")
print(f"Davies-Bouldin Index: {agg_db:.4f}")

=== AGGLOMERATIVE CLUSTERING PERFORMANCE ===
Silhouette Score: 0.2185
Davies-Bouldin Index: 1.3245


## 13. Compare Methods

In [13]:
comparison = pd.DataFrame({
    "Algorithm": [
        "K-Means",
        "Agglomerative Clustering"
    ],

    "Silhouette Score": [
        kmeans_silhouette,
        agg_silhouette
    ],

    "Davies-Bouldin Index": [
        kmeans_db,
        agg_db
    ]
})

display(comparison.round(4))

if kmeans_silhouette > agg_silhouette:
    print("K-Means produced better-separated clusters based on Silhouette Score.")
elif agg_silhouette > kmeans_silhouette:
    print("Agglomerative Clustering produced better-separated clusters based on Silhouette Score.")
else:
    print("Both algorithms produced equal Silhouette Scores.")

,Algorithm,Silhouette Score,Davies-Bouldin Index
0,K-Means,0.2831,1.2701
1,Agglomerative Clustering,0.2185,1.3245


K-Means produced better-separated clusters based on Silhouette Score.


## 14. Sanity Check

Three clients are selected from different parts of the dataset to compare their original values and cluster labels from both methods.

In [14]:
client_indices = [0, 100, 200]

sanity_columns = [
    "Fresh",
    "Milk",
    "Grocery",
    "Frozen",
    "Detergents_Paper",
    "Delicassen",
    "Channel",
    "Region",
    "Cluster",
    "Agg_Cluster"
]

sanity_check = df.loc[
    client_indices,
    sanity_columns
]

print("=== SANITY CHECK: THREE CLIENTS ===")

display(sanity_check)

=== SANITY CHECK: THREE CLIENTS ===


,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Channel,Region,Cluster,Agg_Cluster
0,12669,9656,7561,214,2674,1338,2,3,0,4
100,11594,7779,12144,3252,8035,3029,2,3,2,1
200,3067,13240,23127,3941,9959,731,2,1,4,1


## 15. Save Output

The final saved CSV contains the K-Means cluster labels in the `Cluster` column.

In [15]:
output_df = df.drop(columns=["Agg_Cluster"])

output_df.to_csv(
    "segmented_wholesale_customers.csv",
    index=False
)

print("segmented_wholesale_customers.csv saved successfully.")

print("\nSaved Dataset Shape:", output_df.shape)

display(output_df.head())

segmented_wholesale_customers.csv saved successfully.

Saved Dataset Shape: (440, 9)


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Cluster
0,2,3,12669,9656,7561,214,2674,1338,0
1,2,3,7057,9810,9568,1762,3293,1776,0
2,2,3,6353,8808,7684,2405,3516,7844,0
3,1,3,13265,1196,4221,6404,507,1788,3
4,2,3,22615,5410,7198,3915,1777,5185,3
